In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install numpy matplotlib seaborn scikit-learn tensorflow

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [4]:
dataset_path = '/content/drive/MyDrive/dataset/FER-2013'
train_dir = os.path.join(dataset_path, 'train')
test_dir = os.path.join(dataset_path, 'test')
model_save_dir = '/content/drive/MyDrive/emotion-detect-model'

if not os.path.exists(model_save_dir):
    os.makedirs(model_save_dir)

# Config

In [5]:
batch_size = 64
epochs = 50
image_size = (48, 48)
num_classes = 7


# Tăng cường dữ liệu cho huấn luyện

In [6]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2  # 20% cho kiểm định
)

In [ ]:
# chuẩn hóa cho kiểm tra

In [7]:
test_datagen = ImageDataGenerator(rescale=1./255)

# Generator huấn luyện

In [8]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size,
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

Found 0 images belonging to 7 classes.


# Generator kiểm định

In [9]:
val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size,
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)


Found 0 images belonging to 7 classes.


# Generator kiểm tra

In [10]:
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=image_size,
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # Quan trọng để đánh giá
)


Found 3422 images belonging to 7 classes.


# In thông tin

In [11]:
print(f"Số mẫu huấn luyện: {train_generator.samples}")
print(f"Số mẫu kiểm định: {val_generator.samples}")
print(f"Số mẫu kiểm tra: {test_generator.samples}")
print(f"Chỉ số lớp: {train_generator.class_indices}")

Số mẫu huấn luyện: 0
Số mẫu kiểm định: 0
Số mẫu kiểm tra: 3422
Chỉ số lớp: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


#Define the CNN Model

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(48, 48, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# Set Up Callbacks

In [ ]:
checkpoint = ModelCheckpoint(
    os.path.join(model_save_dir, 'model_epoch_{epoch:02d}_val_acc_{val_accuracy:.4f}.h5'),
    monitor='val_accuracy',
    save_best_only=False,
    mode='auto'
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.0001)

callbacks = [checkpoint, early_stop, reduce_lr]


#Calculate Steps per Epoch



In [ ]:
steps_per_epoch = int(train_generator.samples * 0.2 / batch_size)
validation_steps = val_generator.samples // batch_size

#Train the Model


In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=callbacks
)

# Plot Training History


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Độ chính xác huấn luyện')
plt.plot(history.history['val_accuracy'], label='Độ chính xác kiểm định')
plt.title('Độ chính xác mô hình')
plt.xlabel('Epoch')
plt.ylabel('Độ chính xác')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Mất mát huấn luyện')
plt.plot(history.history['val_loss'], label='Mất mát kiểm định')
plt.title('Mất mát mô hình')
plt.xlabel('Epoch')
plt.ylabel('Mất mát')
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate on Test Set

In [ ]:
# Dự đoán
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

# Tính toán chỉ số
accuracy = accuracy_score(y_true, y_pred_classes)
f1 = f1_score(y_true, y_pred_classes, average='weighted')

print(f"Độ chính xác kiểm tra: {accuracy:.4f}")
print(f"Điểm F1 trung bình có trọng số: {f1:.4f}")

# Ma trận nhầm lẫn
cm = confusion_matrix(y_true, y_pred_classes)
emotion_labels = list(test_generator.class_indices.keys())

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=emotion_labels, yticklabels=emotion_labels)
plt.title('Ma trận nhầm lẫn')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()